# CMSC 173 &middot; Module 6 &middot; Example 4
## Deep or shallow? Philippine earthquakes

From only where an earthquake struck, can a model tell whether it was deep (300 km or more)? 274 of the 11,391 were. A lesson in why accuracy can mislead.

**How to use this notebook.** Runtime &rarr; Run all. Every number printed below is on the
slides; change something and see what moves.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

url = "https://raw.githubusercontent.com/njpinton/cmsc173-labs/main/data/ph_earthquakes.csv"
quakes = pd.read_csv(url, parse_dates=["time"])
quakes["deep"] = (quakes["depth"] >= 300).astype(int)          # 274 deep, 2.4%
X, y = quakes[["longitude", "latitude"]], quakes["deep"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=173, stratify=y)
model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_tr, y_tr)
print(round(model.score(X_te, y_te), 3))      # 0.973
print(confusion_matrix(y_te, model.predict(X_te)))

**97% accurate, and nearly useless.** Guessing *shallow* for every quake scores higher:

In [ ]:
print(round((y_te == 0).mean(), 3))           # 0.976

**One argument fixes it.** `class_weight="balanced"` makes each deep quake count as much as all the shallow ones together.

In [ ]:
bal = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced")).fit(X_tr, y_tr)
print(round(bal.score(X_te, y_te), 3))        # 0.893
print(confusion_matrix(y_te, bal.predict(X_te)))   # finds 73 of 82 deep quakes; 357 false alarms

**Try it.** Add `mag` to `X`. Does it help? (Module 5's last slide has a hint.)